# 04 · 物理仿真与同步多模态数据
执行模块三的 oven4 轨迹，使用同一 MuJoCo 状态生成双相机 RGB、深度、部件 Mask、点云、位姿和帧间事件计数，以及关节、接触和动作序列。

随机化在每个 episode 的种子下重新采样。事件是亮度阈值模拟，物性仍为明确声明的模型先验；这些数据不标为真机采集。

In [ ]:
import json
import uuid

from examples._display import setup_notebook

PROJECT_ROOT, OUTPUT_DIR = setup_notebook("simulation_and_data_generation")

from physrealx import ArtifactRef, PhysRealX, VersionedRef

px = PhysRealX.open(PROJECT_ROOT / ".physrealx")
run_key = uuid.uuid4().hex[:12]


## 1. 场景与任务

In [ ]:
import numpy as np

from physrealx import MethodCapability, RunSpec

upstream = json.loads(
    (PROJECT_ROOT / "examples/scene_and_task_generation/output/results.json").read_text()
)
scene_ref = VersionedRef.model_validate(upstream["scene"])
original_task = px.tasks.get(VersionedRef.model_validate(upstream["task"]))
scene = px.scenes.get(scene_ref)
plan = json.loads(px.artifacts.get_bytes(ArtifactRef.model_validate(upstream["trajectory"])))


## 2. 传感器与随机化

In [ ]:
from physrealx.integrations.sensors import CameraSpec

target = np.array(scene.entities[0].pose.translation_m) + np.array([0, 0, 0.10])
camera_specs = {
    name: CameraSpec.look_at(
        target + offset, target, width=64, height=48, max_depth_m=5.0, event_threshold=0.2
    )
    for name, offset in {"front": [0.85, -0.65, 0.5], "side": [0.6, 0.7, 0.4]}.items()
}
cameras = {name: spec.model_dump(mode="json") for name, spec in camera_specs.items()}
task = px.tasks.with_sensors(
    original_task,
    scene_ref,
    camera_specs,
    resource_id=f"oven4.rendered.{run_key}",
    frequency_hz=20,
)
task_ref = px.tasks.register(task)
backend_config = dict(upstream["backend_config"])
backend_config.update(
    cameras=cameras,
    randomization={
        "mass_scale": [0.8, 1.2],
        "damping_scale": [0.8, 1.2],
        "friction_scale": [0.8, 1.2],
        "initial_position_jitter": 0.005,
        "color_jitter": 0.15,
        "light_jitter": 0.15,
        "camera_position_jitter_m": 0.02,
        "occlusion_probability": 0.4,
        "occlusion_fraction": [0.08, 0.18],
    },
)
backend = px.methods.bind(MethodCapability.SIMULATION_RUN, "mujoco", config=backend_config)
print("同步信号:", len(task.observations), "；相机:", list(cameras))


## 3. 专家示范

In [ ]:
expert_spec = RunSpec(
    run_id=str(uuid.uuid4()),
    scene=scene_ref,
    task=task_ref,
    backend=backend,
    policy=px.methods.bind(
        MethodCapability.POLICY_INFERENCE,
        "joint_sequence",
        config={"commands": plan["commands"]},
    ),
    episode_count=20,
    stop_on_success=False,
    base_seed=20260,
    dataset_id=f"oven4.expert.{run_key}",
    dataset_version="1.0.0",
    record_signals=tuple(signal.name for signal in task.observations),
    tags=("expert_demonstration",),
)
expert_run = px.runs.submit(expert_spec)
expert_result = px.runs.wait(expert_run)
expert_episodes = px.runs.episodes(expert_run)
assert all(episode.success for episode in expert_episodes)
print("成功轨迹:", len(expert_episodes), "帧数:", sum(len(ep.steps) for ep in expert_episodes))


## 4. 失败轨迹

In [ ]:
failure_spec = expert_spec.model_copy(
    update={
        "run_id": str(uuid.uuid4()),
        "dataset_id": f"oven4.failure.{run_key}",
        "base_seed": 30260,
        "episode_count": 6,
        "policy": px.methods.bind(
            MethodCapability.POLICY_INFERENCE, "joint_target", config={"target": [0.0]}
        ),
    }
)
failure_run = px.runs.submit(failure_spec)
failure_result = px.runs.wait(failure_run)
failed_episodes = px.runs.episodes(failure_run)
assert not any(episode.success for episode in failed_episodes)
print("未达目标的真实仿真轨迹:", len(failed_episodes))


## 5. 接触与数据校核
直线执行器使用标准 AssetSpec 和显式物性先验。只有推杆通电驱动，烤箱门没有电机；成功判据读取被动烤箱门位置，接触力必须非零。这是直线执行器推压演示；多指抓取和现场机械臂仍需相应机器人资产与部署接口。


In [ ]:
from physrealx.integrations.datasets import EpisodeCollection, validate_dataset

assert backend.config["controlled_joints"] == ["pusher/extension"]
peak_force = max(
    s.observations["contact.force"].inline[0] for ep in expert_episodes for s in ep.steps
)
assert 0 < peak_force < 100, "被动烤箱门必须由机器人实际接触驱动"
for reference in (expert_result.dataset, failure_result.dataset):
    quality = validate_dataset(px.artifacts, px.datasets.get(reference))
    print(quality)
collection = px.artifacts.put_json(
    EpisodeCollection(datasets=(expert_result.dataset, failure_result.dataset))
)


## 6. 多模态预览

In [ ]:
import matplotlib.pyplot as plt

from physrealx.integrations._artifacts import read_array

sample = expert_episodes[0].steps[len(expert_episodes[0].steps) // 2]
figure, axes = plt.subplots(2, 4, figsize=(12, 5))
for row, name in enumerate(cameras):
    for column, modality in enumerate(("rgb", "depth", "mask", "events")):
        array = read_array(px.artifacts, sample.observations[f"camera.{name}.{modality}"].artifact)
        if modality == "events":
            array = array[:, :, 0] - array[:, :, 1]
        axes[row, column].imshow(array, cmap="viridis" if modality != "rgb" else None)
        axes[row, column].set_title(f"{name} / {modality}")
        axes[row, column].axis("off")
figure.tight_layout()
figure.savefig(OUTPUT_DIR / "multimodal.png", dpi=140)
plt.show()


## 7. 物理状态驱动的外观

读取同一物理资产绑定的 ArtGS 模型，以实际记录的被动烤箱门位置驱动外观，摄像机经过完整位姿变换。此视图单独显示 oven4；双相机全场景、碰撞和多模态数据仍由前面的 MuJoCo 记录提供。超出实拍起止状态的部分按 USD 关节先验外推并标明。

In [ ]:
from physrealx.integrations.gaussian_render import RegisteredRenderInput
from physrealx.integrations.sensors import CameraSpec

oven = px.assets.get(next(entity.asset for entity in scene.entities if entity.name == "oven"))
registration_ref = ArtifactRef.model_validate(oven.metadata["registered_appearance"])
view = dict(cameras["front"])
view.update(width=320, height=240)
trajectory = px.artifacts.put_json(
    RegisteredRenderInput(
        registration=registration_ref,
        episode=px.datasets.get(expert_result.dataset).episodes[0],
        entity="oven",
        camera=CameraSpec.model_validate(view),
        frame_indices=(0, 40, 80, 120, 160),
    )
)
operation = px.renders.generate(trajectory, method="registered_artgs")
assert operation.metrics["physical_motion_span"] > 0.1
assert operation.metrics["foreground_pixels"] > 1000
streams_ref = operation.output("render_streams")
streams = json.loads(px.artifacts.get_bytes(streams_ref))
images = read_array(px.artifacts, ArtifactRef.model_validate(streams["streams"]["rgb"]))
fig, axes = plt.subplots(1, len(images), figsize=(15, 3))
for axis, pixels, position in zip(axes, images, streams["recorded_joint_positions"]):
    axis.imshow(pixels)
    axis.set_title(f"Recorded q = {position:.3f} {streams['joint_position_unit']}")
    axis.axis("off")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "registered-appearance.png", dpi=150)
plt.show()


## 8. 结果导出

In [ ]:
results = {
    "scene": scene_ref.model_dump(mode="json"),
    "task": task_ref.model_dump(mode="json"),
    "expert_dataset": expert_result.dataset.model_dump(mode="json"),
    "failure_dataset": failure_result.dataset.model_dump(mode="json"),
    "episode_collection": collection.model_dump(mode="json"),
    "evaluation_run": expert_spec.model_dump(mode="json"),
    "render_streams": streams_ref.model_dump(mode="json"),
}
results_ref = px.artifacts.export(OUTPUT_DIR, results)
print("结果索引:", OUTPUT_DIR / "results.json")
